In [1]:
print("hi")

hi


Here’s a **complete, clean example of a LightGBM Regressor using MLflow**, including **training, logging, autologging, and model loading**.
You can copy-paste this as a standalone script.

---

## LightGBM Regressor with MLflow (End-to-End)

### 1. Install Dependencies

```bash
pip install mlflow lightgbm scikit-learn
```

---

### 2. Training + MLflow Tracking

```python
import mlflow
import mlflow.lightgbm
import lightgbm as lgb

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# ----------------------------
# Load dataset
# ----------------------------
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------
# MLflow setup
# ----------------------------
mlflow.set_experiment("lgbm_regressor_experiment")
mlflow.lightgbm.autolog()  # logs params, metrics, model automatically

# ----------------------------
# Train model
# ----------------------------
with mlflow.start_run(run_name="lgbm_regressor"):
    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    # ----------------------------
    # Evaluate
    # ----------------------------
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)

    # Manual metrics (optional, autolog already logs some)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
```

---

### 3. View Results

```bash
mlflow ui
```

Open → **[http://localhost:5000](http://localhost:5000)**

You’ll see:

* Parameters (n_estimators, learning_rate, etc.)
* Metrics (rmse, r2)
* Model artifact
* Conda environment

---

### 4. Load the Logged Model

#### Load by Run ID

```python
import mlflow.pyfunc

model = mlflow.pyfunc.load_model(
    "runs:/<RUN_ID>/model"
)

preds = model.predict(X_test)
```

---

### 5. Register the Model (Optional)

```python
mlflow.lightgbm.log_model(
    model,
    artifact_path="model",
    registered_model_name="LGBMRegressor"
)
```

Then promote via UI or API:

```python
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.transition_model_version_stage(
    name="LGBMRegressor",
    version=1,
    stage="Production"
)
```

---

### 6. Serve the Model

```bash
mlflow models serve \
  -m models:/LGBMRegressor/Production \
  -p 5001
```

Request:

```bash
curl -X POST http://localhost:5001/invocations \
  -H "Content-Type: application/json" \
  -d '{"inputs": [[8.3252, 41.0, 6.984, 1.023, 322.0, 2.555, 37.88, -122.23]]}'
```

---

## Recommended LightGBM + MLflow Settings

✅ Use `mlflow.lightgbm.autolog()`
✅ Log **RMSE / MAE / R²** explicitly
✅ Register only **validated** models
✅ Use **separate experiments** per dataset version

---

## Minimal Version (if you want ultra-short)

```python
mlflow.lightgbm.autolog()

with mlflow.start_run():
    model = lgb.LGBMRegressor()
    model.fit(X_train, y_train)
```

---

If you want, I can also provide:

* **Hyperparameter tuning with Optuna + MLflow**
* **Cross-validation logging**
* **Time-series LightGBM**
* **MLflow + S3 / remote tracking**
* **Production-ready folder structure**

Just tell me 👍
